# 2.6 — Random Forest multiclasse (5 classes do motor, `app17-11`)

**Forma 1, não Forma 2:** o `app17-11` já calcula as 12 features no próprio ESP32 e imprime
1 linha CSV por janela — este notebook carrega essas features **prontas** direto (não passa
pelo `2.4`, que só existe para janelar o *raw* da Forma 2). O `2.4` já é agnóstico ao número de
classes (agrupa por `label`/`classe` sem conhecer a lista), então nada nele precisou mudar para
as 5 classes — a reutilização do pipeline de split/validação é o que aparece reimplementado
aqui embaixo, porque aqui não há passagem pelo `2.4` para fazer isso por nós.

**Sequência de classes:** `desligado → operando → inclinado_frente → inclinado_tras → anomalia`
(cíclica). `rodada` identifica cada volta completa pela sequência — é a coluna que usamos no
`LeaveOneGroupOut`, do mesmo jeito que nos notebooks `2.x`.

**O ponto central deste notebook:** o `2.5` concluiu que `mean_ax/ay/az` era um atalho a
descartar (o problema binário não precisava de orientação). Aqui, com `inclinado_frente` e
`inclinado_tras` na mistura, é o contrário — **veja a seção 4**.

In [ ]:
!pip install -q pandas scikit-learn matplotlib micromlgen influxdb-client

## 1) Carregar as janelas — duas fontes possíveis

**Fonte A (padrão):** um arquivo de texto salvo do Monitor Serial durante a coleta (qualquer
forma de salvar a saída serve — "Save Output" do monitor, ou um redirecionamento de terminal).
O parser abaixo é tolerante: filtra só as linhas que batem com as 16 colunas do CSV do
firmware, e descarta o resto (mensagens de boot, `# INICIO`/`# FIM`, linhas em branco).

**Fonte B:** InfluxDB, se você já estiver na Fase MQTT do `app17-11` (ver
`analytics/forma1_nodered/flow_multiclasse_influx.json`) — célula alternativa logo abaixo,
comentada. As duas fontes produzem o mesmo `df`; o resto do notebook não differencia.

In [ ]:
import pandas as pd

COLUNAS = ["classe", "rodada", "janela", "fs_real",
          "mean_ax", "mean_ay", "mean_az", "std_ax", "std_ay", "std_az", "rms_mag",
          "std_mag", "p2p_mag", "crest_mag", "kurt_mag", "zcr_mag"]

try:
    from google.colab import files
    enviados = files.upload()
    nome = list(enviados.keys())[0]
except Exception:
    import glob
    candidatos = glob.glob("*.txt") + glob.glob("*.log") + glob.glob("*.csv")
    nome = candidatos[0] if candidatos else "serial_app17-11.txt"

linhas_validas = []
with open(nome, encoding="utf-8", errors="ignore") as f:
    for linha in f:
        linha = linha.strip()
        if not linha or linha.startswith("#"):
            continue                          # ignora marcadores de rodada
        campos = linha.split(",")
        if len(campos) != len(COLUNAS) or campos[0] == "classe":
            continue                          # ignora boot/cabecalho/linhas soltas
        linhas_validas.append(campos)

df = pd.DataFrame(linhas_validas, columns=COLUNAS)
for c in COLUNAS:
    if c != "classe":
        df[c] = pd.to_numeric(df[c])

print(f"{len(df)} janelas lidas de {nome}")
print(df.groupby(["classe", "rodada"]).size())

### Fonte B — InfluxDB (alternativa, Fase MQTT)

Descomente e preencha se estiver usando o fluxo Node-RED em vez do arquivo salvo do Serial.
Mesmo padrão do notebook `1.3`, measurement `vibracao_multiclasse`.

In [ ]:
# from influxdb_client import InfluxDBClient
#
# INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"
# INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"
# INFLUX_ORG    = "SUA_ORG"
# INFLUX_BUCKET = "sensores"
# MEASUREMENT   = "vibracao_multiclasse"
#
# client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
# flux = f'''
# from(bucket: "{INFLUX_BUCKET}")
#   |> range(start: -30d)
#   |> filter(fn: (r) => r._measurement == "{MEASUREMENT}")
#   |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
# '''
# df = client.query_api().query_data_frame(flux)
# if isinstance(df, list):
#     df = pd.concat(df, ignore_index=True)
# df["rodada"] = df["rodada"].astype(int)
# print(df.groupby(["classe", "rodada"]).size())

## 2) Checar `fs_real` (a taxa de amostragem, medida no próprio ESP32)

Cada janela já vem com `fs_real` — não precisamos estimar pelo timestamp como nos notebooks
`2.x` (lá o timestamp era da recepção no PC; aqui o ESP32 mede a própria taxa,
`amostras / tempo decorrido`). Esperado ≈ 500 Hz.

In [ ]:
FS_NOMINAL = 500  # Hz -- ver 5-REV.2/5-REV.3: provisorio ate reconfirmar o RPM do motor

desvio = (df["fs_real"] - FS_NOMINAL).abs() / FS_NOMINAL
n_fora = (desvio > 0.05).sum()
print(f"fs_real: media={df['fs_real'].mean():.1f} Hz  min={df['fs_real'].min():.1f}  max={df['fs_real'].max():.1f}")
print(f"Janelas com desvio > 5% de {FS_NOMINAL} Hz: {n_fora} de {len(df)}")
if n_fora > 0:
    print(df.loc[desvio > 0.05].groupby(["classe", "rodada"]).size())

## 3) Split treino/teste por rodada (sem vazamento)

Mesma lógica dos notebooks `2.4`/`2.5` — holdout da última rodada de cada classe, com fallback
cronológico 70/30 se só houver 1 rodada. Reimplementada aqui porque o Forma 1 não passa pelo
`2.4` (as features já chegam prontas do ESP32).

In [ ]:
CLASSES = {"desligado": 0, "operando": 1, "inclinado_frente": 2, "inclinado_tras": 3, "anomalia": 4}
NOMES_CLASSES = list(CLASSES.keys())

df = df[df["classe"].isin(CLASSES)].copy()
df["y"] = df["classe"].map(CLASSES)

df["split"] = ""
for classe, g in df.groupby("classe"):
    rodadas = sorted(g["rodada"].unique())
    if len(rodadas) >= 2:
        rodada_teste = rodadas[-1]
        df.loc[g.index[g["rodada"] == rodada_teste], "split"] = "teste"
        df.loc[g.index[g["rodada"] != rodada_teste], "split"] = "treino"
    else:
        ordenado = g.sort_values("janela")
        corte = int(len(ordenado) * 0.7)
        df.loc[ordenado.index[:corte],  "split"] = "treino"
        df.loc[ordenado.index[corte:], "split"] = "teste"

tr = df[df["split"] == "treino"]
te = df[df["split"] == "teste"]
print(f"Treino: {len(tr)} | Teste: {len(te)}")
print(df.groupby(["classe", "rodada", "split"]).size())

## 4) O gancho do notebook `2.5`: mesma feature, veredicto oposto

No `2.5` (normal × anômalo), `mean_ax/ay/az` era um atalho: codificava a *postura* do sensor,
não a vibração, e o modelo que dependesse dele quebraria fora do laboratório. Aqui, com
`inclinado_frente` e `inclinado_tras` — que **só diferem em orientação**, com o motor ligado
nos dois — é o oposto: `mean_*` é a **única família que consegue separar essas duas classes**.
Treinamos o mesmo Random Forest com três conjuntos e olhamos o relatório por classe, não só a
acurácia agregada:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import numpy as np

FEATURES_MEAN = ["mean_ax", "mean_ay", "mean_az"]
FEATURES_AC   = ["std_ax", "std_ay", "std_az", "std_mag", "p2p_mag", "crest_mag", "kurt_mag", "zcr_mag"]
FEATURES_TUDO = FEATURES_MEAN + FEATURES_AC
# 11, nao 12: fica de fora so o rms_mag -- o 2.5 ja mostrou que ele satura na
# gravidade e std_mag carrega a mesma informacao de vibracao sem esse problema.

def treina_avalia(features):
    scaler = StandardScaler().fit(tr[features].values)
    X_tr = scaler.transform(tr[features].values)
    X_te = scaler.transform(te[features].values)
    clf = RandomForestClassifier(n_estimators=30, max_depth=8, random_state=42)
    clf.fit(X_tr, tr["y"].values)
    y_pred = clf.predict(X_te)
    return clf, scaler, y_pred

for nome, feats in [("FEATURES_MEAN (so orientacao, 3)", FEATURES_MEAN),
                    ("FEATURES_AC (sem orientacao, 8)",  FEATURES_AC),
                    ("FEATURES_TUDO (11)",               FEATURES_TUDO)]:
    _, _, y_pred = treina_avalia(feats)
    print(f"=== {nome} ===")
    print(classification_report(te["y"], y_pred, target_names=NOMES_CLASSES,
                                labels=list(range(len(NOMES_CLASSES))), zero_division=0))

**Leitura esperada:** `FEATURES_MEAN` deve acertar bem `inclinado_frente`/`inclinado_tras`
(orientação) mas confundir `desligado`/`operando`/`anomalia` entre si (todas "planas", só
diferem em vibração). `FEATURES_AC` deve ser o oposto: boa nas três "planas", ruim nas duas
inclinações. Nenhum dos dois conjuntos sozinho resolve as 5 — por isso o modelo de produção
(seção 5) usa `FEATURES_TUDO`.

## 5) Modelo final: métricas, matriz de confusão (`FEATURES_TUDO`)

In [ ]:
clf, scaler, y_pred = treina_avalia(FEATURES_TUDO)
acc = accuracy_score(te["y"], y_pred)
f1m = f1_score(te["y"], y_pred, average="macro")
print(f"FEATURES_TUDO: acuracia={acc:.3f}  f1_macro={f1m:.3f}\n")
print(classification_report(te["y"], y_pred, target_names=NOMES_CLASSES,
                            labels=list(range(len(NOMES_CLASSES))), zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(te["y"], y_pred, labels=list(range(len(NOMES_CLASSES)))),
                       display_labels=NOMES_CLASSES).plot(xticks_rotation=45)
plt.title("Matriz de confusao -- FEATURES_TUDO")
plt.tight_layout(); plt.show()

## 6) Validação robusta: `LeaveOneGroupOut` por rodada

Deixamos cada rodada de fora, uma de cada vez — revela se o modelo generaliza entre sessões de
coleta ou decorou uma rodada específica. Precisa de pelo menos 2 rodadas por classe.

In [ ]:
X_all = df[FEATURES_TUDO].values
y_all = df["y"].values
grupos = df["rodada"].astype(str)

n_rodadas_por_classe = df.groupby("classe")["rodada"].nunique()
print("Rodadas por classe:")
print(n_rodadas_por_classe)
if (n_rodadas_por_classe < 2).any():
    print("\nAVISO: alguma classe com menos de 2 rodadas -- colete mais para um resultado confiavel.\n")

logo = LeaveOneGroupOut()
accs, f1s = [], []
for fold, (idx_tr, idx_te) in enumerate(logo.split(X_all, y_all, groups=grupos)):
    scaler_f = StandardScaler().fit(X_all[idx_tr])
    clf_f = RandomForestClassifier(n_estimators=30, max_depth=8, random_state=42)
    clf_f.fit(scaler_f.transform(X_all[idx_tr]), y_all[idx_tr])
    y_pred_f = clf_f.predict(scaler_f.transform(X_all[idx_te]))

    acc_f = accuracy_score(y_all[idx_te], y_pred_f)
    f1_f  = f1_score(y_all[idx_te], y_pred_f, average="macro", zero_division=0)
    rodada_fora = grupos.iloc[idx_te[0]]
    accs.append(acc_f); f1s.append(f1_f)
    print(f"fold {fold+1:2d} (rodada {rodada_fora} fora): acuracia={acc_f:.3f}  f1_macro={f1_f:.3f}")

print(f"\nMedia entre folds: acuracia={np.mean(accs):.3f} (+-{np.std(accs):.3f})  "
     f"f1_macro={np.mean(f1s):.3f}")

## 7) Exportar para o ESP32 (micromlgen)

Exportamos com `FEATURES_TUDO` — diferente do `2.5`, que terminou só com `FEATURES_AC`. Ali
fazia sentido descartar `mean_*` (era um atalho, sem valor físico). Aqui `mean_*` é necessário
de verdade: sem ele, as duas classes de inclinação ficam indistinguíveis (seção 4).

In [ ]:
from micromlgen import port

with open("AIoTMotorMultiClasseRF_micromlgen.hpp", "w") as f:
    f.write(port(clf))

def gerar_scaler_hpp(scaler, features):
    n = len(features)
    means  = ", ".join(f"{m:.10f}f" for m in scaler.mean_)
    scales = ", ".join(f"{s:.10f}f" for s in scaler.scale_)
    lista_features = ", ".join(features)
    return f'''#ifndef STANDARD_SCALER_HPP
#define STANDARD_SCALER_HPP
// StandardScaler de {n} features (FEATURES_TUDO): {lista_features}
namespace Scaler {{
    const static float means[{n}]  = {{ {means} }};
    const static float scales[{n}] = {{ {scales} }};
    inline void std(const float* input, float* output) {{
        for (int i = 0; i < {n}; i++) output[i] = (input[i] - means[i]) / scales[i];
    }}
}}
#endif
'''

with open("AIoTMotorMultiClasseScaler.hpp", "w") as f:
    f.write(gerar_scaler_hpp(scaler, FEATURES_TUDO))

try:
    from google.colab import files
    files.download("AIoTMotorMultiClasseRF_micromlgen.hpp")
    files.download("AIoTMotorMultiClasseScaler.hpp")
except Exception:
    print("Arquivos gerados na pasta atual (fora do Colab).")